# NoSQL Data Modelling with Apache Cassandra

## Project Overview

A music streaming platform (SoundCloud) collects data on songs and user activity but stores everything as CSV files, making it difficult to query efficiently. This project designs and implements a **NoSQL database using Apache Cassandra**, modelling the data to answer three specific analytical queries.

The core principle behind this project is **query-first data modelling** — in Cassandra, tables are designed around the queries they need to serve, not around the data relationships.

---

## The Three Queries

1. Find the **artist name, song title, and song length** heard during `session_number = 338` and `item_in_session = 4`
2. Find the **artist name, song title, and user name** (sorted by item_in_session) for `user_id = 10` and `session_number = 182`
3. Find **every user name** who listened to the song `'All Hands Against His Own'`

## Step 1 — Import Packages

In [1]:
import pandas as pd
import numpy as np
import cassandra
import csv
from cassandra.cluster import Cluster

## Step 2 — Connect to Cassandra & Create Keyspace

A **keyspace** in Cassandra is equivalent to a database/schema in relational databases.
We use `SimpleStrategy` with `replication_factor = 1` suitable for a single-node local setup.

In [2]:
from cassandra.cluster import Cluster

# Connect to local Cassandra instance
try:
    cluster = Cluster(['127.0.0.1'])
    session = cluster.connect()
    print("Successfully connected to the Cassandra cluster.")
except Exception as e:
    print(f"Connection failed: {e}")

# Create keyspace
try:
    session.execute("""
        CREATE KEYSPACE IF NOT EXISTS music_lib
        WITH REPLICATION = { 'class': 'SimpleStrategy', 'replication_factor': 1 }
    """)
    session.set_keyspace('music_lib')
    print("Keyspace 'music_lib' created and set successfully.")
except Exception as e:
    print(f"Error creating keyspace: {e}")

Successfully connected to the Cassandra cluster.
Keyspace 'music_lib' created and set successfully.


---

## Query 1 — Song Details by Session

**Question:** Find the artist name, song title, and song length for `session_number = 338` and `item_in_session = 4`

**Data Modelling Decision:**
- **Partition Key:** `session_number` — groups all rows for a session on the same node
- **Clustering Key:** `item_in_session_number` — orders rows within each partition and allows filtering by both keys
- This combination makes the WHERE clause `session_number = 338 AND item_in_session_number = 4` highly efficient

In [3]:
# Create table — designed specifically for Query 1
try:
    session.execute("""
        CREATE TABLE IF NOT EXISTS song_history (
            session_number         INT,
            item_in_session_number INT,
            artist_name            TEXT,
            song_title             TEXT,
            length_of_song         FLOAT,
            PRIMARY KEY (session_number, item_in_session_number)
        )
    """)
    print("Table 'song_history' created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")

Table 'song_history' created successfully.


In [4]:
# Insert data from CSV
try:
    with open('event_data.csv', encoding='utf8') as f:
        csv_reader = csv.reader(f)
        next(csv_reader)  # skip header
        for row in csv_reader:
            session.execute("""
                INSERT INTO song_history
                    (session_number, item_in_session_number, artist_name, song_title, length_of_song)
                VALUES (%s, %s, %s, %s, %s)
            """, (int(row[8]), int(row[3]), row[0], row[9], float(row[5])))
    print("Data inserted successfully into song_history.")
except Exception as e:
    print(f"Error inserting data: {e}")

Data inserted successfully into song_history.


In [5]:
# Execute Query 1
try:
    rows = session.execute("""
        SELECT artist_name, song_title, length_of_song
        FROM   song_history
        WHERE  session_number = 338
        AND    item_in_session_number = 4
    """)
    print("Query 1 Result:")
    for row in rows:
        print(f"Artist: {row.artist_name}, Song: {row.song_title}, Length: {row.length_of_song:.2f} seconds")
except Exception as e:
    print(f"Query error: {e}")

Query 1 Result:
Artist: Faithless, Song: Music Matters (Mark Knight Dub), Length: 495.31 seconds


---

## Query 2 — User Session Songs (Sorted)

**Question:** Find the artist name, song title (sorted by item_in_session), and user name for `user_id = 10` and `session_number = 182`

**Data Modelling Decision:**
- **Composite Partition Key:** `(user_id, session_number)` — both are needed to uniquely locate the data partition
- **Clustering Key:** `item_in_session_number` — enables ORDER BY sorting within the partition
- Cassandra sorts data by clustering keys at write time, making reads extremely fast

In [6]:
# Create table — designed specifically for Query 2
try:
    session.execute("""
        CREATE TABLE IF NOT EXISTS user_session_songs (
            user_id                INT,
            session_number         INT,
            item_in_session_number INT,
            artist_name            TEXT,
            song_title             TEXT,
            first_name             TEXT,
            last_name              TEXT,
            PRIMARY KEY ((user_id, session_number), item_in_session_number)
        )
    """)
    print("Table 'user_session_songs' created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")

Table 'user_session_songs' created successfully.


In [7]:
# Insert data from CSV
try:
    with open('event_data.csv', encoding='utf8') as f:
        csv_reader = csv.reader(f)
        next(csv_reader)  # skip header
        for row in csv_reader:
            session.execute("""
                INSERT INTO user_session_songs
                    (user_id, session_number, item_in_session_number,
                     artist_name, song_title, first_name, last_name)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
            """, (
                int(row[10]),  # user_id
                int(row[8]),   # session_number
                int(row[3]),   # item_in_session_number
                row[0],        # artist_name
                row[9],        # song_title
                row[1],        # first_name
                row[4]         # last_name
            ))
    print("Data inserted successfully into user_session_songs.")
except Exception as e:
    print(f"Error inserting data: {e}")

Data inserted successfully into user_session_songs.


In [8]:
# Execute Query 2
try:
    rows = session.execute("""
        SELECT   artist_name, song_title, first_name, last_name
        FROM     user_session_songs
        WHERE    user_id = 10
        AND      session_number = 182
        ORDER BY item_in_session_number
    """)
    print("Query 2 Result (sorted by item_in_session):")
    for i, row in enumerate(rows):
        print(f"Item {i} | Artist: {row.artist_name}, Song: {row.song_title}, User: {row.first_name} {row.last_name}")
except Exception as e:
    print(f"Query error: {e}")

Query 2 Result (sorted by item_in_session):
Item 0 | Artist: Down To The Bone, Song: Keep On Keepin' On, User: Sylvie Cruz
Item 1 | Artist: Three Drives, Song: Greece 2000, User: Sylvie Cruz
Item 2 | Artist: Sebastien Tellier, Song: Kilometer, User: Sylvie Cruz
Item 3 | Artist: Lonnie Gordon, Song: Catch You Baby, User: Sylvie Cruz


---

## Query 3 — Users Who Listened to a Song

**Question:** Find every user name who listened to `'All Hands Against His Own'`

**Data Modelling Decision:**
- **Partition Key:** `song_title` — all listeners of a song are stored in the same partition, making the WHERE clause fast
- **Clustering Key:** `user_id` — ensures uniqueness within the partition (a user can only appear once per song)
- We cannot use just `song_title` as the primary key because multiple users can listen to the same song

In [9]:
# Create table — designed specifically for Query 3
try:
    session.execute("""
        CREATE TABLE IF NOT EXISTS song_listeners (
            song_title TEXT,
            user_id    INT,
            first_name TEXT,
            last_name  TEXT,
            PRIMARY KEY (song_title, user_id)
        )
    """)
    print("Table 'song_listeners' created successfully.")
except Exception as e:
    print(f"Error creating table: {e}")

Table 'song_listeners' created successfully.


In [10]:
# Insert data from CSV
try:
    with open('event_data.csv', encoding='utf8') as f:
        csv_reader = csv.reader(f)
        next(csv_reader)  # skip header
        for row in csv_reader:
            session.execute("""
                INSERT INTO song_listeners (song_title, user_id, first_name, last_name)
                VALUES (%s, %s, %s, %s)
            """, (row[9], int(row[10]), row[1], row[4]))
    print("Data inserted successfully into song_listeners.")
except Exception as e:
    print(f"Error inserting data: {e}")

Data inserted successfully into song_listeners.


In [11]:
# Execute Query 3
try:
    rows = session.execute("""
        SELECT first_name, last_name
        FROM   song_listeners
        WHERE  song_title = 'All Hands Against His Own'
    """)
    print("Query 3 Result — Users who listened to 'All Hands Against His Own':")
    for row in rows:
        print(f"{row.first_name} {row.last_name}")
except Exception as e:
    print(f"Query error: {e}")

Query 3 Result — Users who listened to 'All Hands Against His Own':
Jacqueline Lynch
Tegan Levine
Sara Johnson


---

## Step 3 — Drop Tables & Close Connection

Clean up all tables and close the Cassandra session and cluster connection.

In [12]:
# Drop all tables
tables = ['song_history', 'user_session_songs', 'song_listeners']

for table in tables:
    try:
        session.execute(f"DROP TABLE IF EXISTS {table}")
        print(f"Table dropped: {table}")
    except Exception as e:
        print(f"Error dropping {table}: {e}")

Table dropped: song_history
Table dropped: user_session_songs
Table dropped: song_listeners


In [13]:
# Close connections
session.shutdown()
cluster.shutdown()
print("Cassandra session and cluster connection closed.")

Cassandra session and cluster connection closed.
